# Notebook 18 — operational diagnostic capture recovery

The original diagnostic registered 256 patch tokens but captured 256-pixel images with the frozen model’s 8-pixel patches, producing 1,024 tokens. Its 18 UNKNOWN cases and all checkpoints remain preserved. This separate adapter changes only the capture resolution to `16 × patch_size = 128`, restoring the intended 16×16 grid. It uses the same four TRAIN images, frozen seed/model, ranks, feature bases, temperatures and numerical case logic.

The original diagnostic activation deadline **13:09:56 UTC on 12 October 2026** is inherited; no phase clock resets or extra GPU budget. This CPU-only detached recovery may therefore have only a few minutes remaining. Official images are not read. Negative diagnostic fidelity never blocks the independently running learned architecture.

Outputs: `artifacts/notebook_18/learned-kernel-001/diagnostic_recovery/` registration, JSON, logs and portable report. Checkpoints: `checkpoints/learned-kernel-001/diagnostic_recovery/`, latest two and best with original source/clock, RNG, case position and captured state. A short checkpoint/source smoke runs before launch. The original scientific Notebook 18 exports remain unchanged.

Exact resume: Run All in this notebook. The `resume-diagnostic-recovery` cell launches/reuses `fmm18-diagnostic-recovery-001`; attach with `tmux attach -t fmm18-diagnostic-recovery-001`, detach with Ctrl+b then d. The original deadline and immutable adapter/parent registration remain required. Invalid state raises rather than silently restarting; interrupted cases remain UNKNOWN. This notebook does not restart or control the full learning workers.


In [ ]:
from pathlib import Path
import copy,fcntl,hashlib,json,os,random,subprocess,sys,time,types,shlex
import numpy as np
import torch
ROOT=Path(os.environ.get('FMM_JEPA_ROOT','/home/benjamin/workspace/ahr_go/fmm_jeppa')).resolve()
if '__file__'not in globals():__file__=str(ROOT/'experiments/18_operational_recovery/diagnostic_capture.ipynb')
SCHEMA='fmm18-operational-diagnostic-capture-recovery/v1'
RUN_ID='learned-kernel-001'
ARTIFACT=ROOT/'artifacts/notebook_18'/RUN_ID/'diagnostic_recovery'
SESSION='fmm18-diagnostic-recovery-001'


def diagnostic(root=ROOT):
    path=Path(root)/'experiments/18_learned_kernel/diagnostics.ipynb';module=types.ModuleType('fmm18_operational_original_diagnostic');module.__file__=str(path)
    for cell in json.loads(path.read_text())['cells']:
        if cell['cell_type']=='code'and'ahr-export'in cell.get('metadata',{}).get('tags',[]):exec(compile(''.join(cell['source']),str(path)+'#ahr-export','exec'),module.__dict__)
    return module


def export_hash(path):
    nb=json.loads(Path(path).read_text());source='\n'.join(''.join(c['source'])for c in nb['cells']if c['cell_type']=='code'and'ahr-export'in c.get('metadata',{}).get('tags',[]))
    return hashlib.sha256(source.encode()).hexdigest()


def corrected_capture_function(D):
    """Two explicit operational substitutions; every other frozen definition remains identical.

    Original image256 with patch8 yields1024 tokens. Registered diagnosticN256
    requires a16x16 patch grid, hence image128. The numerical case logic, source
    model, local radius, tau, basis seeds, ranks, masks and IDs remain unchanged.
    """
    nb=json.loads(Path(D.__file__).read_text());source='\n'.join(''.join(c['source'])for c in nb['cells']if c['cell_type']=='code'and'ahr-export'in c.get('metadata',{}).get('tags',[]))
    start=source.index('def _capture(');end=source.index('\ndef save_checkpoint(',start);original=source[start:end]
    old1="changed['image_size']=256";old2="grid=256//int(config['model']['patch_size'])"
    if original.count(old1)!=1 or original.count(old2)!=1:raise ValueError('Exact frozen capture substitution sites changed')
    corrected=original.replace(old1,"changed['image_size']=16*int(config['model']['patch_size'])").replace(old2,'grid=16')
    namespace=dict(D.__dict__);exec(compile(corrected,str(Path(__file__))+'#ahr-export-corrected-capture','exec'),namespace)
    return namespace['_capture'],{'original_capture_definition_sha256':hashlib.sha256(original.encode()).hexdigest(),'corrected_capture_definition_sha256':hashlib.sha256(corrected.encode()).hexdigest(),'substitutions':{old1:"changed['image_size']=16*int(config['model']['patch_size'])",old2:'grid=16'}}


def registration(root=ROOT):
    root=Path(root);D=diagnostic(root);study=root/'artifacts/notebook_18'/RUN_ID;original=study/'diagnostic';artifact=study/'diagnostic_recovery';artifact.mkdir(parents=True,exist_ok=True)
    original_request=D.read(original/'diagnostic_registration.json');clock=D.read(original/'diagnostic_clock.json');activation=D.read(study/'diagnostic_activation.json');error=D.read(original/'diagnostic_result.json')
    if error['status']!='UNKNOWN' or len(error['rows'])!=18 or any(row['status']!='UNKNOWN_CASE_ERROR'for row in error['rows']):raise ValueError('Expected preserved originalN256 capture failure evidence required')
    if clock['deadline_unix']!=activation['deadline_unix']:raise ValueError('Original phase and activation deadlines disagree')
    if D.source_hashes(root,'full')!=original_request['source_hashes']:raise ValueError('Original scientific source graph changed')
    original_capture=D.read(original/'capture_result.json')['capture_record']
    if D.file_record(original_capture['path'])!=original_capture:raise ValueError('Original capture checksum changed')
    capture=torch.load(original_capture['path'],map_location='cpu',weights_only=False)
    if any(layer['tokens'].shape[1]!=1024 or tuple(layer['grid'])!=(32,32)for layer in capture['layers'].values()):raise ValueError('Unexpected original capture mismatch')
    _,patch=corrected_capture_function(D)
    value={'schema':SCHEMA,'run_id':RUN_ID,'scientific_source_hashes':original_request['source_hashes'],'adapter':{'path':str(Path(__file__).resolve()),'export_sha256':export_hash(__file__)},
      'original_registration_record':D.file_record(original/'diagnostic_registration.json'),'original_error_result_record':D.file_record(original/'diagnostic_result.json'),
      'original_clock_record':D.file_record(original/'diagnostic_clock.json'),'parent_activation_record':D.file_record(study/'diagnostic_activation.json'),'parent_clock':clock,
      'original_capture_record':original_capture,'sample_ids':capture['sample_ids'],'source_checkpoint':capture['source']['checkpoint'],
      'capture_patch_size':8,'capture_image_size':128,'tokens':256,'grid':[16,16],'reason':'Restore registeredN256 scope: image_size=16*actual patch_size8=128. Original image256 produced1024 tokens and is retained asUNKNOWN.',
      'ranks':original_request['ranks'],'basis_seeds':original_request['basis_seeds'],'layers':sorted(map(int,capture['layers'])),
      'deadline_unix':float(clock['deadline_unix']),'no_reset_or_additional_budget':True,'official_test_pixels_read':0,'learning_blocked_by_fidelity':False,'capture_definition_patch':patch}
    return D.immutable(artifact/'recovery_registration.json',value)


def save(D,folder,identity,completed,capture_record=None,active=None):
    manifest=D.save_checkpoint(folder,identity,completed,capture_record,active)
    manifest['resume_notebook']='experiments/18_operational_recovery/diagnostic_capture.ipynb';manifest['resume_cell']='resume-diagnostic-recovery';D.atomic_json(Path(folder)/'manifest.json',manifest);return manifest


def smoke(root=ROOT):
    root=Path(root);D=diagnostic(root);reg=registration(root);artifact=root/'artifacts/notebook_18'/RUN_ID/'diagnostic_recovery';folder=root/'checkpoints'/RUN_ID/'diagnostic_recovery_validation';checks=[]
    check=lambda name,condition:checks.append({'name':name,'passed':True})if condition else(_ for _ in()).throw(AssertionError(name))
    check('Recovered resolution implies registered16x16 tokens',(reg['capture_image_size']//reg['capture_patch_size'])**2==reg['tokens'])
    check('Original900s clock is inherited',reg['deadline_unix']==reg['parent_clock']['deadline_unix'] and reg['parent_clock']['deadline_unix']-reg['parent_clock']['started_unix']<=900.)
    identity={'schema':SCHEMA,'validation':'SYNTHETIC_RNG_CHECKPOINT_ONLY','adapter':reg['adapter'],'parent_clock':reg['parent_clock']}
    state=D.rng_state();save(D,folder,identity,{});expected=(random.random(),float(np.random.random()),torch.rand(5));D.restore_checkpoint(folder,identity);actual=(random.random(),float(np.random.random()),torch.rand(5))
    check('Exact checkpoint next Python NumPy Torch RNG',expected[0]==actual[0]and expected[1]==actual[1]and torch.equal(expected[2],actual[2]))
    for i in range(3):save(D,folder,identity,{str(j):{'status':'UNKNOWN_SYNTHETIC_CHECKPOINT_FIXTURE'}for j in range(i+1)})
    manifest=D.read(folder/'manifest.json');check('Retain latest2 and best',len(manifest['latest_two'])==2 and len(list(folder.glob('diagnostic-*.pt')))<=3)
    check('Latest2 and best checkpoint hashes valid',all(D.file_record(record['path'])==record for record in manifest['latest_two']+[manifest['best']]))
    wrong=copy.deepcopy(identity);wrong['adapter']['export_sha256']='changed'
    try:D.restore_checkpoint(folder,wrong)
    except ValueError:checks.append({'name':'Changed source identity refused','passed':True})
    else:raise AssertionError('Changed identity accepted')
    D.restore_rng(state)
    check('Scientific18 source graph unchanged',D.source_hashes(root,'full')==reg['scientific_source_hashes'])
    receipt={'schema':SCHEMA,'status':'PASS','checks':checks,'check_count':len(checks),'role':'SYNTHETIC_CHECKPOINT_AND_SOURCE_VALIDATION','adapter':reg['adapter'],'registration_record':D.file_record(artifact/'recovery_registration.json'),'official_test_pixels_read':0}
    D.atomic_json(artifact/'smoke_receipt.json',receipt);return receipt


def run(root=ROOT):
    root=Path(root);artifact=root/'artifacts/notebook_18'/RUN_ID/'diagnostic_recovery';artifact.mkdir(parents=True,exist_ok=True);D=diagnostic(root)
    with(artifact/'single_writer.lock').open('a')as handle:
        fcntl.flock(handle,fcntl.LOCK_EX|fcntl.LOCK_NB)
        reg=registration(root);identity={'schema':SCHEMA,'registration':reg,'registration_sha256':D.digest(reg)};limit=reg['deadline_unix'];folder=root/'checkpoints'/RUN_ID/'diagnostic_recovery'
        completed,capture_record,active=D.restore_checkpoint(folder,identity)
        if active is not None:
            completed.setdefault(active,{'key':active,'status':'UNKNOWN_INTERRUPTED_CASE','reason':'RecoveryIN_FLIGHT preserved without retry'});save(D,folder,identity,completed,capture_record)
        if capture_record is None:
            if 'FROZEN_TRAIN_CAPTURE'in completed:raise RuntimeError('Interrupted recovery capture is never silently restarted')
            D.check_deadline(limit,5);save(D,folder,identity,completed,None,'FROZEN_TRAIN_CAPTURE');capture_fn,_=corrected_capture_function(D)
            capture_record,capture=capture_fn(root,artifact,'full',limit)
            if capture['sample_ids']!=reg['sample_ids']or capture['source']['checkpoint']!=reg['source_checkpoint']:raise ValueError('Recovered TRAIN IDs or source model differ')
            if any(tuple(layer['grid'])!=(16,16)or layer['tokens'].shape[1]!=256 for layer in capture['layers'].values()):raise ValueError('Recovery capture still violatesN256')
            capture['source']['operational_recovery']={'adapter':reg['adapter'],'registration_sha256':identity['registration_sha256'],'actual_capture_image_size':128,'original_failure_preserved':True}
            D.atomic_torch(capture_record['path'],capture);capture_record=D.file_record(capture_record['path']);D.atomic_json(artifact/'capture_result.json',{'status':'CAPTURED','capture_record':capture_record,'adapter':reg['adapter'],'original_deadline_unix':limit,'official_test_pixels_read':0})
            save(D,folder,identity,completed,capture_record)
        else:capture=torch.load(capture_record['path'],map_location='cpu',weights_only=False)
        jobs=[(layer,rank,seed)for layer in reg['layers']for rank in reg['ranks']for seed in reg['basis_seeds']]
        for layer,rank,seed in jobs:
            key=f'l{layer}-r{rank}-s{seed}'
            if key in completed:continue
            if time.time()+3>=limit:completed[key]={'key':key,'layer':layer,'rank':rank,'feature_seed':seed,'status':'UNKNOWN_GLOBAL_DEADLINE'};save(D,folder,identity,completed,capture_record);continue
            job={'key':key,'profile':'full','layer':layer,'rank':rank,'feature_seed':seed,'capture_record':capture_record,'source_hashes':reg['scientific_source_hashes'],'request_sha256':identity['registration_sha256'],'deadline':limit,'result_path':str(artifact/'cases'/key/'result.json')}
            job_path=artifact/'cases'/key/'job.json';D.immutable(job_path,job);save(D,folder,identity,completed,capture_record,key)
            row=D._subprocess_case(root,job_path,min(120.,limit-time.time()));row.update(layer=layer,rank=rank,feature_seed=seed);completed[key]=row;save(D,folder,identity,completed,capture_record)
            D.atomic_json(artifact/'recovery_progress.json',{'status':'RUNNING','completed_count':len(completed),'requested_cases':18,'deadline_unix':limit,'rows':list(completed.values()),'adapter':reg['adapter']})
        rows=list(completed.values());complete=len(rows)==18 and all(row.get('status')=='MEASURED'for row in rows)
        result={'schema':SCHEMA,'status':'COMPLETE'if complete else'UNKNOWN','profile':'TRAIN_ONLY_OPERATIONAL_CAPTURE_RECOVERY','rows':rows,'summary':D.summarize(rows),'registration':reg,
          'registration_record':D.file_record(artifact/'recovery_registration.json'),'capture_record':capture_record,'parent_clock':reg['parent_clock'],'deadline_unix':limit,
          'adapter':reg['adapter'],'original_error_result_record':reg['original_error_result_record'],'original_failure_preserved':True,'scientific_source_graph_unchanged':D.source_hashes(root,'full')==reg['scientific_source_hashes'],
          'official_test_pixels_read':0,'learning_blocked_by_fidelity':False,'finished_unix':time.time()}
        D.atomic_json(artifact/'recovery_result.json',result)
        report=D.load(root,'experiments/18_learned_kernel/report.ipynb','fmm18_operational_report')
        report.render(root,artifact,{'status':result['status'],'phase':'TRAIN_ONLY_OPERATIONAL_DIAGNOSTIC_RECOVERY','reason':'Corrected originalN256 image capture. OriginalUNKNOWN retained; classification/training results remain in the main18 report.','original_error_result_record':reg['original_error_result_record']},diagnostic=result)
        return result


def launch(root=ROOT):
    root=Path(root);artifact=root/'artifacts/notebook_18'/RUN_ID/'diagnostic_recovery';D=diagnostic(root);receipt=smoke(root);reg=registration(root)
    probe=subprocess.run(['tmux','has-session','-t',SESSION],capture_output=True)
    if probe.returncode==0:return {'status':'ALREADY_RUNNING','session':SESSION,'deadline_unix':reg['deadline_unix'],'artifact':str(artifact)}
    code="import json; from pathlib import Path; p=Path("+repr(str(Path(__file__).resolve()))+"); ns={'__file__':str(p)}; [exec(compile(''.join(c['source']),str(p)+'#ahr-export','exec'),ns)for c in json.loads(p.read_text())['cells']if c['cell_type']=='code'and'ahr-export'in c.get('metadata',{}).get('tags',[])]; ns['run']()"
    command='env CUDA_VISIBLE_DEVICES= OMP_NUM_THREADS=2 MKL_NUM_THREADS=2 '+shlex.quote(sys.executable)+' -u -c '+shlex.quote(code)+' > '+shlex.quote(str(artifact/'coordinator.log'))+' 2>&1'
    subprocess.run(['tmux','new-session','-d','-s',SESSION,command],check=True)
    result={'status':'LAUNCHED','session':SESSION,'attach':'tmux attach -t '+SESSION,'detach':'Ctrl+b then d','deadline_unix':reg['deadline_unix'],'original_clock_preserved':True,'CPU_ONLY':True,
      'artifact':str(artifact),'checkpoint':str(root/'checkpoints'/RUN_ID/'diagnostic_recovery'),'adapter':reg['adapter'],'validation_receipt':D.file_record(artifact/'smoke_receipt.json'),'launched_unix':time.time()}
    D.atomic_json(artifact/'launch_receipt.json',result);return result


### Resume the bounded CPU recovery

In [ ]:
launch(ROOT)